# Common Compute Pattern — Visualize

Per-beam generated-token counts for two problems, side by side.

- **Model:** Qwen2.5-1.5B-Instruct
- **Dataset:** MATH-500
- **Beams:** n = 4

**Algorithms shown:**
1. `best_of_n` — read `best_of_n/beams=4.csv` and plot `gen_len` per beam for two chosen `problem_idx` values.
2. `beam_search` — read the per-step trace `beam_search_per_step_{method}_n4.csv`, fix `step_idx = 0` (first step), and plot `gen_len` per beam for the same two problems.

In each subplot the bar with the largest value is highlighted in `BAR_COLOR_DARK`; the rest use `BAR_COLOR`. Charts are saved to `../figures/` as `compute_pattern_visualize_best_of_n.png` and `compute_pattern_visualize_beam_search.png`.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- Config ---------------------------------------------------------------
MODEL_DIR    = 'QWen2.5-1.5B-Instruct'
DATASET_DIR  = 'MATH-500'
N_BEAMS      = 4
PROBLEM_IDS  = [0, 1]      # the two problem indices to visualize
BS_STEP_IDX  = 0           # 'first step' for beam_search
BS_METHOD    = 'ours'      # 'ours' | 'vllm' for beam_search per-step trace

BAR_WIDTH      = 0.5               # narrower bars (matplotlib default is 0.8)
BAR_COLOR      = '#87d6c1'  # default bar color
BAR_COLOR_DARK = '#3fb797'       # highlight color for the tallest bar in each subplot

# ---- Paths ----------------------------------------------------------------
PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data' / 'Qwen' / MODEL_DIR / DATASET_DIR

BON_CSV      = DATA_DIR / 'best_of_n' / f'beams={N_BEAMS}.csv'
BS_CSV       = DATA_DIR / 'raw_experiment_data_beam_search' / f'beam_search_per_step_{BS_METHOD}_n{N_BEAMS}.csv'

FIGURES_DIR  = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert BON_CSV.exists(), f'missing: {BON_CSV}'
assert BS_CSV.exists(),  f'missing: {BS_CSV}'
print('best_of_n CSV:  ', BON_CSV)
print('beam_search CSV:', BS_CSV)
print('figures dir:    ', FIGURES_DIR)

def bar_colors(vals: np.ndarray) -> list[str]:
    """Dark color for the (first) tallest bar, light for the rest."""
    top = int(np.argmax(vals))
    return [BAR_COLOR_DARK if i == top else BAR_COLOR for i in range(len(vals))]

## 1. `best_of_n` — gen_len per beam, two problems

In [ ]:
bon = pd.read_csv(BON_CSV)
# Schema: problem_idx, beam_idx, gen_len

def bon_beams(problem_idx: int) -> np.ndarray:
    sub = bon[bon['problem_idx'] == problem_idx].sort_values('beam_idx')
    # Reindex to beams 0..N_BEAMS-1 so missing beams show as 0 rather than collapse.
    arr = np.zeros(N_BEAMS, dtype=float)
    for _, row in sub.iterrows():
        b = int(row['beam_idx'])
        if 0 <= b < N_BEAMS:
            arr[b] = row['gen_len']
    return arr

fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.7), sharey=True)
x_labels = [f'#{i + 1}' for i in range(N_BEAMS)]
x = np.arange(N_BEAMS)

for ax, pid in zip(axes, PROBLEM_IDS):
    vals = bon_beams(pid)
    bars = ax.bar(x, vals, width=BAR_WIDTH, color=bar_colors(vals))
    ax.bar_label(bars, fmt='%.0f', fontsize=12, padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=12)
    ax.set_xlabel('beams', fontsize=14)
    ax.tick_params(axis='y', labelsize=12)
    # ax.set_title(f'best_of_n | problem_idx={pid}')
    ax.set_ylim(0, 840)
    ax.set_xlim(-0.5, 3.5)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

axes[0].set_ylabel('Generation length', fontsize=14)
# fig.suptitle(f'best_of_n | {DATASET_DIR} | {MODEL_DIR} | n={N_BEAMS}', y=1.02)
plt.tight_layout()

out_path = FIGURES_DIR / 'compute_pattern_visualize_best_of_n.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print('saved:', out_path)
plt.show()

## 2. `beam_search` — gen_len per beam at the first step (`step_idx = 0`)

The per-step trace's `problem` column is a string like `"p0: <prompt prefix>..."`. We extract the leading `pNN` to recover the problem index.

In [ ]:
bs = pd.read_csv(BS_CSV)
# Schema: problem, step_idx, beam_idx, gen_len, gen_time_ms, completed

_pid_re = re.compile(r'^p(\d+):')
bs['problem_idx'] = bs['problem'].str.extract(_pid_re, expand=False).astype(int)

def bs_beams(problem_idx: int, step_idx: int = BS_STEP_IDX) -> np.ndarray:
    sub = bs[(bs['problem_idx'] == problem_idx) & (bs['step_idx'] == step_idx)].sort_values('beam_idx')
    arr = np.zeros(N_BEAMS, dtype=float)
    for _, row in sub.iterrows():
        b = int(row['beam_idx'])
        if 0 <= b < N_BEAMS:
            arr[b] = row['gen_len']
    return arr

fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.7), sharey=True)
x = np.arange(N_BEAMS)

for ax, pid in zip(axes, PROBLEM_IDS):
    vals = bs_beams(pid)
    bars = ax.bar(x, vals, width=BAR_WIDTH, color=bar_colors(vals))
    ax.bar_label(bars, fmt='%.0f', fontsize=12, padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=12)
    ax.set_xlabel('beams', fontsize=14)
    ax.tick_params(axis='y', labelsize=12)
    # ax.set_title(f'beam_search | problem_idx={pid} | step={BS_STEP_IDX}')
    ax.set_ylim(0, 230)
    ax.set_xlim(-0.5, 3.5)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

axes[0].set_ylabel('Generation length', fontsize=14)
# fig.suptitle(f'beam_search ({BS_METHOD}) | {DATASET_DIR} | {MODEL_DIR} | n={N_BEAMS}', y=1.02)
plt.tight_layout()

out_path = FIGURES_DIR / 'compute_pattern_visualize_beam_search.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print('saved:', out_path)
plt.show()